# Week 00 — Setup Check

**MCTA 4364 Machine Vision** · self-study before Week 1 · about 15 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W00_setup/W00_setup_check.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W00_setup/W00_setup_check.ipynb)

Run this notebook **once** on the computer you will use for the labs. It checks that every tool the course needs is installed
and working, and tells you what your hardware can do. If anything shows ❌, follow the hint or see [SETUP.md](../../SETUP.md).

| Where you run the notebooks | What you need to do |
|---|---|
| **Google Colab** | Nothing. For Weeks 6–13 choose *Runtime → Change runtime type → T4 GPU*. |
| **Kaggle** | Turn on *Internet* and choose a *GPU* accelerator in the notebook settings. |
| **Your laptop** | Follow [SETUP.md](../../SETUP.md): Python, PyTorch with CUDA, then `pip install -r requirements.txt`. |

## 1. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

## 2. Python packages

In [ ]:
import importlib
import platform
import shutil

from cvhelpers import SMOKE

print(f"Python {platform.python_version()} on {platform.system()} {platform.release()}\n")
packages = [("numpy", "core"), ("cv2", "core"), ("skimage", "core"), ("matplotlib", "core"), ("scipy", "core"),
            ("ipywidgets", "core"), ("plotly", "core"), ("PIL", "core"), ("torch", "weeks 6-13"),
            ("torchvision", "weeks 6-13"), ("ultralytics", "weeks 8-11"), ("transformers", "weeks 12-13"),
            ("diffusers", "week 13")]
ok = {}
for name, needed_for in packages:
    try:
        mod = importlib.import_module(name)
        ok[name] = True
        print(f"✅ {name:13s} {getattr(mod, '__version__', ''):14s} ({needed_for})")
    except Exception as exc:  # noqa: BLE001
        ok[name] = False
        print(f"❌ {name:13s} {'missing':14s} ({needed_for}) -> pip install {'opencv-python' if name == 'cv2' else 'scikit-image' if name == 'skimage' else 'pillow' if name == 'PIL' else name}")
total, used, free = shutil.disk_usage(".")
print(f"\nFree disk space: {free / 1e9:.1f} GB (datasets and model weights need up to ~25 GB over the semester)")

## 3. GPU check
The deep-learning weeks run much faster on an NVIDIA GPU. The course is designed so that a **4 GB** laptop GPU is enough.

In [ ]:
if ok.get("torch"):
    import torch
    sys.path.append("resources/scripts")
    from dlhelpers import hw_profile
    HW = hw_profile()
    if HW["cuda"]:
        import time
        x = torch.randn(2048, 2048, device="cuda")
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(20):
            y = x @ x
        torch.cuda.synchronize()
        gflops = 20 * 2 * 2048 ** 3 / (time.time() - t0) / 1e9
        print(f"✅ Matrix-multiply speed on the GPU: {gflops:,.0f} GFLOP/s")
        if HW["vram_gb"] < 3.5:
            print("⚠️ Less than 4 GB of GPU memory: use the smaller settings suggested in each notebook.")
    else:
        print("ℹ️ No NVIDIA GPU detected. Everything still works on the CPU, only slower. "
              "On a laptop with an NVIDIA GPU, reinstall PyTorch with CUDA (SETUP.md step 2).")
else:
    print("PyTorch is not installed yet: it is only needed from Week 6. See SETUP.md step 2.")

## 4. Downloading course data
The notebooks download sample images, datasets and model weights on first use into the `data/` folder.

In [ ]:
from cvhelpers import load_sample, show

img = load_sample("fruits.jpg")
print("✅ Downloaded and loaded an OpenCV sample image:", img.shape)
show(img, titles=["OpenCV sample image 'fruits.jpg'"])

## 5. Interactive widgets and 3D plots
Move the slider: the image should update. Then rotate the 3D plot with your mouse.

In [ ]:
import cv2
import numpy as np
import plotly.graph_objects as go

from cvhelpers import interact


def blur_demo(kernel=1):
    show(cv2.GaussianBlur(img, (2 * kernel + 1, 2 * kernel + 1), 0), titles=[f"Gaussian blur, kernel {2 * kernel + 1}"])


interact(blur_demo, kernel=(0, 15))
gray = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (80, 60))
fig = go.Figure(go.Surface(z=gray, colorscale="gray"))
fig.update_layout(title="Your first interactive 3D plot: image brightness as a surface", height=450,
                  scene=dict(zaxis_title="brightness"))
fig.show()

## 6. Webcam (laptop only)
Several labs use your laptop camera. Set `TEST_WEBCAM = True` and run the cell. Colab and Kaggle cannot access a local webcam this way.

In [ ]:
TEST_WEBCAM = False

if TEST_WEBCAM:
    cap = cv2.VideoCapture(0)
    ok_cam, frame = cap.read()
    cap.release()
    if ok_cam:
        print(f"✅ Webcam works: {frame.shape[1]}x{frame.shape[0]} pixels")
        show(frame, titles=["Webcam snapshot"])
    else:
        print("❌ No webcam frame: close other apps using the camera or try VideoCapture(1).")

## 7. You are ready
If sections 2–5 show ✅ (PyTorch may still be missing until Week 6), you are ready for **Week 01A**.

**Useful links:** [SETUP.md](../../SETUP.md) · [SYLLABUS.md](../../SYLLABUS.md) · [Colab GPU guide](https://research.google.com/colaboratory/faq.html)